# Options Pricing: Black-Scholes vs Binomial vs Monte Carlo

In [ ]:
import sys
sys.path.insert(0, '..')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import os

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
import warnings
warnings.filterwarnings('ignore')

# Ensure output directories exist
os.makedirs('../results/figures', exist_ok=True)
os.makedirs('../results/tables', exist_ok=True)

from src.pricing import BlackScholes, binomial_european, binomial_american, monte_carlo_european, benchmark_methods
from src import config

print('All imports successful.')

# 1. Parameters

In [ ]:
# Common parameters
S = 100       # Spot price
K = 100       # Strike price (ATM)
T = 0.25      # Time to expiry (3 months)
r = 0.05      # Risk-free rate
q = 0         # Continuous dividend yield
sigma = 0.20  # Volatility (20%)

print(f'Spot (S):       {S}')
print(f'Strike (K):     {K}')
print(f'Expiry (T):     {T} years ({T*365:.0f} days)')
print(f'Risk-free (r):  {r:.2%}')
print(f'Dividend (q):   {q:.2%}')
print(f'Volatility:     {sigma:.2%}')

# 2. Black-Scholes Pricing

In [ ]:
bs = BlackScholes(S, K, T, r, q, sigma)

call_price = float(bs.call_price())
put_price = float(bs.put_price())

print('Black-Scholes Analytical Prices')
print('=' * 40)
print(f'Call price: ${call_price:.6f}')
print(f'Put price:  ${put_price:.6f}')
print(f'd1 = {float(bs.d1):.6f}')
print(f'd2 = {float(bs.d2):.6f}')

# 3. Put-Call Parity Verification

In [ ]:
lhs = call_price - put_price
rhs = S * np.exp(-q * T) - K * np.exp(-r * T)
parity_error = abs(lhs - rhs)

print('Put-Call Parity Check')
print('=' * 50)
print(f'C - P                          = {lhs:.8f}')
print(f'S*exp(-qT) - K*exp(-rT)        = {rhs:.8f}')
print(f'Absolute error                 = {parity_error:.2e}')
print(f'\nParity holds: {"YES" if parity_error < 1e-10 else "NO"} (tolerance: 1e-10)')

# 4. Binomial Tree Convergence

In [ ]:
N_values = [10, 25, 50, 100, 200, 500, 1000]
bin_call_prices = []
bin_put_prices = []
bin_errors = []

for N in N_values:
    bp_call = binomial_european(S, K, T, r, q, sigma, N=N, option_type='call')
    bp_put = binomial_european(S, K, T, r, q, sigma, N=N, option_type='put')
    bin_call_prices.append(bp_call)
    bin_put_prices.append(bp_put)
    bin_errors.append(abs(bp_call - call_price))
    print(f'N={N:>5d}:  Call={bp_call:.6f}  Put={bp_put:.6f}  Error={abs(bp_call - call_price):.6f}')

print(f'\nBS reference: Call={call_price:.6f}  Put={put_price:.6f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: binomial prices vs N
ax = axes[0]
ax.plot(N_values, bin_call_prices, 'o-', color='steelblue', label='Binomial Call', markersize=6)
ax.axhline(y=call_price, color='crimson', linestyle='--', linewidth=1.5, label=f'BS Call = {call_price:.4f}')
ax.set_xlabel('Number of Steps (N)', fontsize=12)
ax.set_ylabel('Option Price ($)', fontsize=12)
ax.set_title('Binomial Price Convergence', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xscale('log')

# Right panel: absolute error vs N (log-log)
ax = axes[1]
ax.plot(N_values, bin_errors, 's-', color='darkorange', markersize=6)
ax.set_xlabel('Number of Steps (N)', fontsize=12)
ax.set_ylabel('Absolute Error ($)', fontsize=12)
ax.set_title('Convergence Error (log-log)', fontsize=13, fontweight='bold')
ax.set_xscale('log')
ax.set_yscale('log')

# Fit convergence rate
log_N = np.log(N_values)
log_err = np.log(bin_errors)
slope, intercept = np.polyfit(log_N, log_err, 1)
ax.plot(N_values, np.exp(intercept) * np.array(N_values)**slope, '--', color='gray',
        label=f'Slope = {slope:.2f} (O(1/N))', linewidth=1.5)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('../results/figures/binomial_convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/binomial_convergence.png')

# 5. American vs European: Early Exercise Premium

In [ ]:
N_steps = 500

# European vs American calls
euro_call = binomial_european(S, K, T, r, q, sigma, N=N_steps, option_type='call')
amer_call = binomial_american(S, K, T, r, q, sigma, N=N_steps, option_type='call')

# European vs American puts
euro_put = binomial_european(S, K, T, r, q, sigma, N=N_steps, option_type='put')
amer_put = binomial_american(S, K, T, r, q, sigma, N=N_steps, option_type='put')

print(f'Binomial Tree (N={N_steps})')
print('=' * 55)
print(f'{"":>12s}  {"European":>12s}  {"American":>12s}  {"Premium":>12s}')
print(f'{"Call":>12s}  {euro_call:>12.6f}  {amer_call:>12.6f}  {amer_call - euro_call:>12.6f}')
print(f'{"Put":>12s}  {euro_put:>12.6f}  {amer_put:>12.6f}  {amer_put - euro_put:>12.6f}')
print(f'\nPut early exercise premium: ${amer_put - euro_put:.6f}')
print(f'Call early exercise premium: ${amer_call - euro_call:.6f} (expected ~ 0 for q=0)')

# Demonstrate with dividend yield
q_div = 0.03
euro_call_div = binomial_european(S, K, T, r, q_div, sigma, N=N_steps, option_type='call')
amer_call_div = binomial_american(S, K, T, r, q_div, sigma, N=N_steps, option_type='call')
print(f'\nWith q={q_div:.0%}:')
print(f'  European call: ${euro_call_div:.6f}')
print(f'  American call: ${amer_call_div:.6f}')
print(f'  Early exercise premium: ${amer_call_div - euro_call_div:.6f}')

# 6. Monte Carlo Pricing

In [ ]:
mc_result = monte_carlo_european(S, K, T, r, q, sigma, option_type='call',
                                  n_paths=500_000, antithetic=True, control_variate=True)

print('Monte Carlo European Call (500K paths, antithetic + control variate)')
print('=' * 65)
print(f'Price:           ${mc_result["price"]:.6f}')
print(f'Standard Error:  ${mc_result["standard_error"]:.6f}')
print(f'95% CI:          [{mc_result["ci_lower"]:.6f}, {mc_result["ci_upper"]:.6f}]')
print(f'BS reference:    ${call_price:.6f}')
print(f'Error vs BS:     ${mc_result["price"] - call_price:.6f}')
print(f'\nBS price within 95% CI: {"YES" if mc_result["ci_lower"] <= call_price <= mc_result["ci_upper"] else "NO"}')

# 7. Monte Carlo Convergence

In [ ]:
n_paths_list = [1000, 5000, 10000, 50000, 100000, 500000]
mc_prices = []
mc_ses = []
mc_ci_lowers = []
mc_ci_uppers = []

for n in n_paths_list:
    res = monte_carlo_european(S, K, T, r, q, sigma, option_type='call',
                               n_paths=n, antithetic=True, control_variate=True, seed=42)
    mc_prices.append(res['price'])
    mc_ses.append(res['standard_error'])
    mc_ci_lowers.append(res['ci_lower'])
    mc_ci_uppers.append(res['ci_upper'])
    print(f'N={n:>7,d}: Price={res["price"]:.6f}, SE={res["standard_error"]:.6f}, '
          f'CI=[{res["ci_lower"]:.4f}, {res["ci_upper"]:.4f}]')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Price convergence with confidence intervals
ax = axes[0]
ax.errorbar(n_paths_list, mc_prices,
            yerr=[np.array(mc_prices) - np.array(mc_ci_lowers),
                  np.array(mc_ci_uppers) - np.array(mc_prices)],
            fmt='o-', color='steelblue', capsize=4, markersize=5, label='MC Price +/- 95% CI')
ax.axhline(y=call_price, color='crimson', linestyle='--', linewidth=1.5, label=f'BS = {call_price:.4f}')
ax.set_xlabel('Number of Paths', fontsize=12)
ax.set_ylabel('Option Price ($)', fontsize=12)
ax.set_title('MC Price Convergence', fontsize=13, fontweight='bold')
ax.set_xscale('log')
ax.legend(fontsize=10)

# Right: Standard error convergence (log-log)
ax = axes[1]
ax.plot(n_paths_list, mc_ses, 's-', color='darkorange', markersize=6, label='Standard Error')
# Theoretical 1/sqrt(N) reference
ref_se = mc_ses[0] * np.sqrt(n_paths_list[0]) / np.sqrt(n_paths_list)
ax.plot(n_paths_list, ref_se, '--', color='gray', label=r'$O(1/\sqrt{N})$ reference')
ax.set_xlabel('Number of Paths', fontsize=12)
ax.set_ylabel('Standard Error ($)', fontsize=12)
ax.set_title('Standard Error Convergence', fontsize=13, fontweight='bold')
ax.set_xscale('log')
ax.set_yscale('log')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('../results/figures/mc_convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/mc_convergence.png')

# 8. Variance Reduction Comparison

In [ ]:
n_mc = 100_000
configs = [
    ('Plain MC',           False, False),
    ('Antithetic',         True,  False),
    ('Control Variate',    False, True),
    ('Antithetic + CV',    True,  True),
]

vr_results = []
for label, anti, cv in configs:
    res = monte_carlo_european(S, K, T, r, q, sigma, option_type='call',
                               n_paths=n_mc, antithetic=anti, control_variate=cv, seed=42)
    vr_results.append({
        'Method': label,
        'Price': res['price'],
        'SE': res['standard_error'],
        'Error vs BS': res['price'] - call_price,
    })
    print(f'{label:>20s}: Price={res["price"]:.6f}, SE={res["standard_error"]:.6f}')

vr_df = pd.DataFrame(vr_results)

# Variance reduction factor relative to Plain MC
base_se = vr_df.loc[0, 'SE']
vr_df['Variance Reduction Factor'] = (base_se / vr_df['SE']) ** 2
print('\n', vr_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

colors = ['#4c72b0', '#55a868', '#c44e52', '#8172b2']
bars = ax.bar(vr_df['Method'], vr_df['SE'], color=colors, edgecolor='black', linewidth=0.8)

# Add value labels on bars
for bar, se_val in zip(bars, vr_df['SE']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.0005,
            f'{se_val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Standard Error ($)', fontsize=12)
ax.set_title(f'Variance Reduction Comparison ({n_mc:,} paths)', fontsize=13, fontweight='bold')
ax.set_ylim(0, vr_df['SE'].max() * 1.3)

plt.tight_layout()
plt.savefig('../results/figures/variance_reduction_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/variance_reduction_comparison.png')

# 9. Method Benchmark

In [ ]:
benchmark_df = benchmark_methods(S, K, T, r, q, sigma, option_type='call')
print(benchmark_df.to_string(index=False))

# Save benchmark results
benchmark_df.to_csv('../results/tables/pricing_benchmark.csv', index=False)
print('\nSaved: results/tables/pricing_benchmark.csv')

In [ ]:
# Comprehensive pricing comparison across moneyness levels
moneyness_strikes = [80, 90, 95, 100, 105, 110, 120]
comparison_rows = []

for K_i in moneyness_strikes:
    bs_i = BlackScholes(S, K_i, T, r, q, sigma)
    bs_call = float(bs_i.call_price())
    bs_put = float(bs_i.put_price())
    bin_call = binomial_european(S, K_i, T, r, q, sigma, N=500, option_type='call')
    bin_put = binomial_european(S, K_i, T, r, q, sigma, N=500, option_type='put')
    mc_call = monte_carlo_european(S, K_i, T, r, q, sigma, option_type='call', n_paths=200_000, seed=42)
    mc_put = monte_carlo_european(S, K_i, T, r, q, sigma, option_type='put', n_paths=200_000, seed=42)
    amer_put = binomial_american(S, K_i, T, r, q, sigma, N=500, option_type='put')

    comparison_rows.append({
        'Strike': K_i,
        'Moneyness': K_i / S,
        'BS Call': round(bs_call, 4),
        'BS Put': round(bs_put, 4),
        'Bin Call': round(bin_call, 4),
        'Bin Put': round(bin_put, 4),
        'MC Call': round(mc_call['price'], 4),
        'MC Put': round(mc_put['price'], 4),
        'Amer Put': round(amer_put, 4),
        'Early Ex Premium': round(amer_put - bs_put, 4),
    })

comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))

comparison_df.to_csv('../results/tables/pricing_comparison.csv', index=False)
print('\nSaved: results/tables/pricing_comparison.csv')

In [ ]:
# Summary of files saved
saved_files = [
    'results/figures/binomial_convergence.png',
    'results/figures/mc_convergence.png',
    'results/figures/variance_reduction_comparison.png',
    'results/tables/pricing_benchmark.csv',
    'results/tables/pricing_comparison.csv',
]

print('Files saved by this notebook:')
print('=' * 50)
for f in saved_files:
    print(f'  {f}')